In [1]:
# imports

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window, hilbert
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import importlib
import utils as ut
importlib.reload(ut)

<module 'utils' from '/Users/itamarmerfeld/Projects/UATR/dev/utils.py'>

In [4]:
file_path = '../data/ds2/pos/dpv1_1m.wav'
data, fs = librosa.load(file_path)

fft_nperseg= 8000
percent_overlap=0.25
window='hann'
remove_dc=20
crop_freq=2000
normalization_window_size=9

F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc, crop_freq=crop_freq)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=normalization_window_size)


fig = make_subplots(rows=1, cols=2, column_widths=[0.8, 0.2], column_titles=["Spectrogram", "Spectrum"], shared_yaxes=True)
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', name='Normalized Spectrum', marker_color='Blue', showlegend=False), row=1, col=2)
fig.update_layout(height=600, width=1000, font=dict(size=20))
fig.update_xaxes(title_text='Time (s)', title_font=dict(size=20), col=1, row=1)
fig.update_yaxes(title_text='Frequency (Hz)', title_font=dict(size=20), col=1, row=1)
fig.update_xaxes(title_text='Norm. Power', title_font=dict(size=20), col=2, row=1)
fig.update_yaxes(title_text='Frequency (Hz)', title_font=dict(size=20), col=2, row=1)
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/plots/dpv_analysis.pdf", scale=10)
fig.show()